# Part 3 — Evaluation & Guardrails: is the agent good *and* safe?

Parts 1 and 2 built an agent that gives fluent, confident answers. Here is
what no demo tells you: **a fluent answer and a correct answer look exactly
the same.** An agent that invents a price, recommends a sold-out product,
or describes an item you do not sell will say it just as smoothly as it
says something true.

Before this agent goes near a real customer, one question has to be
answered: *how often is it right — and how do you know?*

This notebook builds the harness that answers it.

> **Before you start:** [`../setup.md`](../setup.md) done, and
> `python ../llm.py` prints `connection ok`.

## The one idea

Evaluation is not complicated. It is three steps:

1. **A fixed set of test questions** — `eval_dataset.json`, 20 of them, each
   with a short note of what a good answer should do.
2. **Run the agent over all of them**, and check every answer two ways:
   - **Guardrails** — deterministic Python checks for things that must
     *never* happen: an invented product, a wrong price, a sold-out item
     sold as available. Pass or fail. No LLM, no cost, no judgement call.
   - **An LLM-as-judge** — a second model call that scores how *helpful*
     the answer is, 1 to 5. The soft, qualitative half.
3. **A scorecard** — pass rates, an average quality score, every failure
   listed.

Guardrails answer *"is it safe?"*. The judge answers *"is it good?"*. You
need both — and you need them as a number, not a feeling.

In [ ]:
import json
import re
import sys
from pathlib import Path

sys.path.append("..")  # so we can import llm.py from the repo root
from llm import EXTRA_BODY, embed_client, embed_config, embed_extra_body, llm_config, raw_client

import faiss
import numpy as np

# The test set, and the structured facts the guardrails check answers against.
EVAL_SET = json.loads(Path("eval_dataset.json").read_text(encoding="utf-8"))
products = json.loads(
    Path("../part-1-build-an-agent/products.json").read_text(encoding="utf-8")
)
FACTS = {p["product_id"]: p for p in products}

CHAT_MODEL = llm_config()["model"]
EMBED_MODEL = embed_config()["model"]

# Retrieval over the Part 2 catalogue (condensed — see Part 2 for the walkthrough).
catalog = [
    {"id": path.stem, "text": path.read_text(encoding="utf-8")}
    for path in sorted(Path("../part-2-agentic-rag/catalog").glob("*.md"))
]
_emb = embed_client().embeddings.create(
    model=EMBED_MODEL,
    input=[doc["text"] for doc in catalog],
    extra_body=embed_extra_body("passage"),
)
_vectors = np.array(
    [item.embedding for item in sorted(_emb.data, key=lambda x: x.index)],
    dtype="float32",
)
faiss.normalize_L2(_vectors)
_index = faiss.IndexFlatIP(_vectors.shape[1])
_index.add(_vectors)

print(f"{len(EVAL_SET)} test questions | {len(FACTS)} products | retrieval index ready")

## The system under test

You cannot evaluate nothing — so first, the thing being evaluated. Here is
the Part 2 retrieval assistant in compact form: retrieve the most relevant
products, then answer from them. This is our *system under test*; everything
after this cell measures **its** output.

In [ ]:
def retrieve(query: str, k: int = 3) -> list[str]:
    """Return the text of the k catalogue documents closest to `query`."""
    response = embed_client().embeddings.create(
        model=EMBED_MODEL, input=[query], extra_body=embed_extra_body("query")
    )
    vec = np.array([response.data[0].embedding], dtype="float32")
    faiss.normalize_L2(vec)
    _, ids = _index.search(vec, k)
    return [catalog[int(i)]["text"] for i in ids[0]]


SYSTEM_PROMPT = """You are a shopping assistant for an outdoor-gear shop.
Answer the customer using only the product documents provided. Cite every
product you mention by its id in square brackets, e.g. [JKT-001], and give
its price in CHF. If a product is out of stock, say so. Never invent
products, prices or stock — if nothing fits the request, say so honestly."""


def shop_assistant(question: str) -> str:
    """The agent under test: retrieve, then answer."""
    context = "\n\n---\n\n".join(retrieve(question))
    reply = raw_client().chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"PRODUCT DOCUMENTS:\n{context}\n\nCUSTOMER: {question}"},
        ],
        temperature=0.1,
        extra_body=EXTRA_BODY,
    )
    return reply.choices[0].message.content or "(empty response)"


print(shop_assistant("I need a warm jacket for cold, dry winter weather."))

## Layer 1 — Guardrails: the things that must never happen

Some mistakes are not "low quality" — they are unacceptable. A shop agent
that quotes the wrong price has not given a 3-out-of-5 answer; it has
mis-sold. These checks are **deterministic**: plain Python, no LLM, instant,
free. Each one reads an answer and returns a list of problems — an empty
list means it passed.

Three guardrails. Notice how small they are: the most important safety
check for a shop is a handful of lines, not a framework.

In [ ]:
ID_RE = re.compile(r"\b[A-Z]{3}-\d{3}\b")
PRICE_RE = re.compile(r"CHF\s*([0-9]+(?:\.[0-9]+)?)")
OUT_OF_STOCK_PHRASES = (
    "out of stock", "sold out", "unavailable", "not available",
    "not in stock", "currently unavailable",
)


def check_real_ids(answer: str) -> list[str]:
    """Every product id the answer cites must exist in the catalogue."""
    return [pid for pid in sorted(set(ID_RE.findall(answer))) if pid not in FACTS]


def check_prices(answer: str) -> list[str]:
    """A price stated next to a product id must match that product's real price."""
    problems = []
    for line in answer.splitlines():
        prices = [float(p) for p in PRICE_RE.findall(line)]
        for pid in ID_RE.findall(line):
            if pid in FACTS:
                real = FACTS[pid]["price_chf"]
                problems += [
                    f"{pid}: answer says CHF {stated:g}, catalogue says CHF {real:g}"
                    for stated in prices
                    if stated != real
                ]
    return problems


def check_stock_honesty(answer: str) -> list[str]:
    """If the answer cites an out-of-stock product, it must say it is unavailable."""
    flagged = any(phrase in answer.lower() for phrase in OUT_OF_STOCK_PHRASES)
    return [
        f"{pid} is out of stock, but the answer does not say so"
        for pid in sorted(set(ID_RE.findall(answer)))
        if pid in FACTS and not FACTS[pid]["in_stock"] and not flagged
    ]


GUARDRAILS = {
    "invented product ids": check_real_ids,
    "price accuracy": check_prices,
    "out-of-stock honesty": check_stock_honesty,
}


def run_guardrails(answer: str) -> dict[str, list[str]]:
    """Run every guardrail. All-empty lists means the answer passed."""
    return {name: check(answer) for name, check in GUARDRAILS.items()}


print("Guardrails ready:", list(GUARDRAILS))

## Watch a guardrail catch a hallucination

Guardrails are only worth anything if they actually fire. Here is an answer
with three planted mistakes — a wrong price, a sold-out item sold as "in
stock", and a product that does not exist. The checks should catch all
three.

In [ ]:
bad_answer = (
    "Great choice! I recommend the Stormshield Hardshell [JKT-001] at "
    "CHF 149 — it is in stock and ships today. You might also like our "
    "Glacier Dome 4-Person Tent [TNT-009]."
)
print("ANSWER UNDER TEST:")
print(" ", bad_answer, "\n")

for name, problems in run_guardrails(bad_answer).items():
    print(f"  [{'FAIL' if problems else 'ok'}] {name}")
    for problem in problems:
        print(f"         - {problem}")

All three caught — a wrong price (JKT-001 is CHF 289, not 149), a sold-out
item ([JKT-001]) presented as available, and an invented product ([TNT-009]
does not exist). No model, no cost, no ambiguity.

## Layer 2 — LLM-as-judge: is the answer any good?

Guardrails catch what is *wrong*. They cannot tell you whether an answer is
*helpful* — whether it actually addressed the customer. That is a judgement
call, so we hand it to a model: a second LLM call that scores the answer
1–5 against a per-question rubric.

The judge prompt is right here in the open — no framework, nothing hidden.
Read it: that prompt *is* the evaluation criterion.

In [ ]:
def judge(question: str, expected: str, answer: str) -> dict[str, object]:
    """LLM-as-judge: score an answer's quality from 1 to 5."""
    prompt = (
        "You are reviewing a shopping assistant's answer to a customer.\n\n"
        f"CUSTOMER QUESTION:\n{question}\n\n"
        f"WHAT A GOOD ANSWER SHOULD DO:\n{expected}\n\n"
        f"THE ANSWER:\n{answer}\n\n"
        "Rate the answer from 1 (poor) to 5 (excellent) on how helpful, "
        "relevant and clear it is for the customer. Reply with one line of "
        'JSON and nothing else: {"score": <1-5>, "reason": "<one sentence>"}'
    )
    reply = raw_client().chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        extra_body=EXTRA_BODY,
    )
    text = reply.choices[0].message.content or ""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return {"score": None, "reason": f"unparseable judge reply: {text[:60]}"}
    try:
        verdict = json.loads(match.group())
        return {"score": int(verdict["score"]), "reason": str(verdict.get("reason", ""))}
    except (ValueError, KeyError, TypeError) as exc:
        return {"score": None, "reason": f"unparseable judge reply: {exc}"}


_demo = EVAL_SET[0]
_demo_answer = shop_assistant(_demo["question"])
print("Q :", _demo["question"])
print("A :", _demo_answer)
print()
print("Judge:", judge(_demo["question"], _demo["expected"], _demo_answer))

## The evaluation loop

Everything is in place. The loop is the obvious thing: for each test
question, get an answer, run the guardrails, ask the judge — and collect it
all.

This makes two model calls per question (one answer, one judge) plus a
retrieval — about 60 calls for 20 questions, comfortably inside the free
tier. It takes a couple of minutes.

In [ ]:
results = []
for item in EVAL_SET:
    answer = shop_assistant(item["question"])
    guardrails = run_guardrails(answer)
    verdict = judge(item["question"], item["expected"], answer)

    safe = all(not problems for problems in guardrails.values())
    results.append(
        {"item": item, "answer": answer, "guardrails": guardrails, "verdict": verdict}
    )
    print(f"  {item['id']}  guardrails {'pass' if safe else 'FAIL'}   quality {verdict['score']}/5")

print(f"\nEvaluated {len(results)} answers.")

In [ ]:
line = "=" * 58
print(line)
print(f"  EVALUATION SCORECARD  —  {len(results)} questions")
print(line)

print("\nGuardrails  (safety — every failure is a mis-sold customer):")
for name in GUARDRAILS:
    clean = sum(1 for r in results if not r["guardrails"][name])
    print(f"  {name:.<28} {clean}/{len(results)} clean")

scores = [r["verdict"]["score"] for r in results if r["verdict"]["score"] is not None]
print("\nQuality  (LLM-as-judge, 1-5):")
if scores:
    print(f"  average score ............... {sum(scores) / len(scores):.2f}")
    spread = "  ".join(f"{s}:{scores.count(s)}" for s in range(5, 0, -1))
    print(f"  distribution ................ {spread}")

print("\nFailures:")
found = False
for r in results:
    rid = r["item"]["id"]
    for name, problems in r["guardrails"].items():
        for problem in problems:
            found = True
            print(f"  [{rid}] guardrail · {name} — {problem}")
    score = r["verdict"]["score"]
    if score is not None and score <= 2:
        found = True
        print(f"  [{rid}] quality {score}/5 — {r['verdict']['reason']}")
if not found:
    print("  none — every answer passed the guardrails and scored 3 or better.")

## Reading the scorecard — and what it does not cover

Your scorecard probably came back clean: guardrails all green, quality
close to 5, no failures listed. That feels like success. Here is the most
important sentence in this notebook — **a clean scorecard on 20 questions
is not a safe agent; it is the absence of evidence on 20 questions.**

What the scorecard genuinely tells you:

- **The guardrails work.** You saw them catch a wrong price, a sold-out
  item and an invented product. When the agent eventually does one of those
  for real — and over thousands of live queries it will — this harness
  catches it.
- **No regression on the cases you thought of.** Re-run this after every
  prompt change and a drop shows up immediately. That is real value.

What it does **not** tell you:

- **Quality near 5 is partly the judge being generous.** One small model
  grading 1–5 tends to mark high. Treat the quality score as a thermometer
  for *change*, not a certificate — which is exactly why the objective,
  pass/fail guardrails carry more weight than the average.
- **20 questions is a sample, not coverage.** Real evaluation sets grow
  every time a customer finds a new way to break the agent. Twenty green
  rows only mean you have not yet tested the twenty-first.
- **Guardrails only catch what they were written to catch.** The price and
  stock checks are heuristics that read text near a product id; a
  differently-shaped hallucination slips straight past.

---

**From demo to production — the whole tutorial in one line.** Across three
parts you built an agent (Part 1), grounded it in a catalogue (Part 2), and
measured it (Part 3). Every part ended at the same honest border: a demo
that works on 12 products and 20 questions is not an agent you put in front
of paying customers. Continuous evaluation against a growing test set,
guardrails wired *into* the live system so a bad answer is blocked rather
than counted, a real catalogue, deployment, Swiss data-protection — that is
the engineering work between this notebook and a shop assistant you can
trust with revenue. It is exactly the work [MEMOTECH](https://memotech.ch)
does.